# 01. LangChain 기초


> LangChain 언어 모델에 기반한 AI 애플리케이션을 쉽게 개발할 수 있도록 도와주는 프레임워크
>> 기존에 오픈AI와 같은 언어 모델의 API를 사용해 원하는 기능을 구현하려면 모든 코드를 직접 작성해야 했으나, 랭체인은 이 작업을 간소화할 수 있는 다양한 도구와 모듈 제공

>  **OpenAI** `.env`의 `OPENAI_API_KEY` 그대로 사용

---
## 0. 설치

In [58]:
!pip install openai python-dotenv langchain langchain-openai langchain-core

---
## 1. Day 3 OpenAI SDK vs LangChain

| | Day 3 (OpenAI SDK) | Day 5 (LangChain) |
|---|---|---|
| 호출 | `client.chat.completions.create()` | `llm.invoke()` |
| 메시지 | `{'role':'user','content':...}` | `HumanMessage`, `SystemMessage` |
| 체인 | 직접 함수 조합 | **LCEL** (`\|` 파이프) |
| RAG/Agent | 직접 루프 작성 | Retriever · AgentExecutor |

---
## 2. ChatOpenAI 연결

In [2]:
from pathlib import Path
from dotenv import load_dotenv
import os


load_dotenv(Path('..') / '.env') 


True

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

# 문자열만 넘겨도 됨
out = llm.invoke('LangChain이 뭐야? 한 문장으로.') #invoke 로 하면 됨. 

In [4]:
out

AIMessage(content='LangChain은 자연어 처리(NLP) 모델을 활용하여 다양한 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 18, 'total_tokens': 51, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_86eb24cbce', 'id': 'chatcmpl-DuzfO1xrYx47Dr0bTj0X4C4kEk6Vu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f03d6-08d1-7102-9982-15e96574599d-0', usage_metadata={'input_tokens': 18, 'output_tokens': 33, 'total_tokens': 51, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [5]:
out.content #메시지 내용 추출

'LangChain은 자연어 처리(NLP) 모델을 활용하여 다양한 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.'

In [6]:
#정석
from langchain_core.messages import HumanMessage


message = [HumanMessage(content = "Langchain이 뭐야? 한 문장으로.")]
out = llm.invoke(message)

In [7]:
out

AIMessage(content='Langchain은 자연어 처리(NLP) 모델을 활용하여 다양한 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 18, 'total_tokens': 51, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_59e007eafa', 'id': 'chatcmpl-DuzfQO7MxOsTATo0uXQHmZeq6TLuF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f03d6-11cb-77e0-84c5-6ffa45ed983f-0', usage_metadata={'input_tokens': 18, 'output_tokens': 33, 'total_tokens': 51, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [8]:
out.content

'Langchain은 자연어 처리(NLP) 모델을 활용하여 다양한 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.'

---
## 3. Message 타입 (Day 3 messages 리스트와 동일 역할)

In [9]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 원래 했던 방식
# messages = [  
#     {"role": "system", "content": '너는 Agent 수업 도우미야. 한국어로 간결하게 답해.'},  # 초기 시스템 메시지
#     {"role": "user", "content" : 'RAG가 뭐야? 2문장으로 설명해.'} # 사용자 메시지
# ]

messages = [
    SystemMessage(content='너는 Agent 수업 도우미야. 한국어로 간결하게 답해.'), # 초기 시스템 메시지
    HumanMessage(content='RAG가 뭐야? 2문장으로 설명해.'), # 사용자 메시지
]


In [10]:
# 과거 방식, 한번에 하는건 드래그 컨트롤 슬래쉬
# def get_ai_response(messages):
#     response = client.chat.completions.create(
#         model="gpt-4o",  # 응답 생성에 사용할 모델 지정
#         temperature=0.9,  # 응답 생성에 사용할 temperature 설정
#         messages=messages,  # 대화 기록을 입력으로 전달
#     )
#     return response.choices[0].message.content  # 생성된 응답의 내용 반환

# answer = get_ai_response(messages)

# 랭체인 방식
answer = llm.invoke(messages) #answer는 ai 메시지
print(answer.content)
print('타입:', type(answer))

RAG는 Retrieval-Augmented Generation의 약자로, 정보 검색과 생성 모델을 결합한 기술입니다. 이 방법은 외부 데이터베이스에서 정보를 검색하여 더 정확하고 풍부한 응답을 생성하는 데 사용됩니다.
타입: <class 'langchain_core.messages.ai.AIMessage'>


멀티턴 실습 lanchain_multiturn.py : 기존 open ai api -> langchain으로 교체

---
## 4. 메시지 히스토리 (멀티턴)

기존에는 멀티턴 대화를 위해 사용자의 대화 내용을 리스트나 딕셔너리에 추가하는 코드 작성

-> 메시지 히스토리 (Message History) 기능으로 쉽게 구현 가능

In [11]:
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼wrapper 클래스
from langchain_openai import ChatOpenAI  # 오픈AI 모델을 사용하는 랭체인 챗봇 클래스
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-4o-mini")

In [12]:
# 세션별 대화 기록을 저장할 딕셔너리
store = {}

In [13]:
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

In [14]:
# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history)

/opt/homebrew/Caskroom/miniconda/base/envs/day3/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3550: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [15]:
store

{}

In [16]:
config = {"configurable": {"session_id": "first"}}  # 세션 ID를 설정하는 config 객체 생성, first 부분은 바꾸면서 사용 가능

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 신범교 라고 해.")],
    config=config,
)

print(response.content)

안녕하세요, 신범교님! 어떻게 도와드릴까요?


In [19]:
store #store에 내용이 생김

{'first': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DuzfbQ5R8sSTDNC7bokjjfb3dFEn3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f03d6-3c58-7b52-beb7-873334919764-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])}

In [20]:
store['first']

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DuzfbQ5R8sSTDNC7bokjjfb3dFEn3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f03d6-3c58-7b52-beb7-873334919764-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])

In [21]:
get_session_history('first')

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DuzfbQ5R8sSTDNC7bokjjfb3dFEn3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f03d6-3c58-7b52-beb7-873334919764-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])

In [22]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 신범교입니다. 어떻게 도와드릴까요?


In [23]:
store['first']

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DuzfbQ5R8sSTDNC7bokjjfb3dFEn3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f03d6-3c58-7b52-beb7-873334919764-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMessage(content='내 이름이 뭐지?', additional_kwargs={

In [24]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 신범교님! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 18, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-DuzfbQ5R8sSTDNC7bokjjfb3dFEn3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f03d6-3c58-7b52-beb7-873334919764-0', usage_metadata={'input_tokens': 18, 'output_tokens': 15, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}),
 AIMessa

In [38]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 신범교님입니다. 다른 궁금한 점이 있으시면 언제든지 말씀해 주세요!


In [39]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 18, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-Dutms4pBdcCs5pejlSOuBCoVacCdf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-2bd1-7a80-95ed-24590210cbb4-0', usage_metadata={'input_tokens': 18, 'output_tokens': 21, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}

In [40]:
config = {"configurable": {"session_id": "second"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

죄송하지만, 당신의 이름을 알 수 있는 정보는 없습니다. 제가 도와드릴 수 있는 다른 것이 있다면 말씀해 주세요!


In [41]:
store

{'first': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 18, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-Dutms4pBdcCs5pejlSOuBCoVacCdf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-2bd1-7a80-95ed-24590210cbb4-0', usage_metadata={'input_tokens': 18, 'output_tokens': 21, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMessage(content='내 이름이 뭐지?'

In [42]:
config = {"configurable": {"session_id": "first"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

귀하의 이름은 신범교님입니다. 추가로 궁금하거나 필요한 것이 있으시면 말씀해 주세요!


In [43]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 18, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_30f4302bf1', 'id': 'chatcmpl-Dutms4pBdcCs5pejlSOuBCoVacCdf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-2bd1-7a80-95ed-24590210cbb4-0', usage_metadata={'input_tokens': 18, 'output_tokens': 21, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}

In [ ]:
first"}}
for r in with_message_history.stream(# 스트림 방식으로 출력
config = {"configurable": {"session_id": "
    [HumanMessage(content = "내 이름이 뭔지 말하고 이름의 뜻을 유추해서 말해봐 ")],
    config=config):
    print(r.content, end="||") # 


||귀||하||의|| 이름||은|| 신||범||교||입니다||.|| 이름||의|| 각|| 요소||를|| 분석||해||보||면||:

||-|| **||신||(||新||)**||:|| 새||롭||다||,|| 신||성||하다||
||-|| **||범||(||範||)**||:|| 본||보기||,|| 귀||감||,|| 범||위||
||-|| **||교||(||敎||)**||:|| 가||르||침||,|| 교육||

||이||렇게|| 보면|| "||신||범||교||"||라는|| 이름||은|| "||새||로운|| 가||르||침||의|| 본||보기||"|| 또는|| "||새||로운|| 교육||의|| 귀||감||"||이라는|| 의미||로|| 해||석||할|| 수|| 있을|| 것|| 같습니다||.|| 이는|| 아||마||도|| 신||범||교||님||이|| 많은|| 사람||들에게|| 긍||정||적인|| 영향을|| 주||고||,|| 새로운|| 것을|| 전||파||하는|| 역할||을|| 하||기를|| 바||라는|| 마음||이|| 담||겼||을|| 수|| 있습니다||.|| 

||혹||시|| 다른|| 질문||이나|| 궁||금||한|| 사항||이|| 있으||시면|| 말씀||해|| 주세요||!||||||

---
## 5. LCEL — LangChain Expression Language

LCEL은 랭체인에서 복잡한 작업 흐름을 간단하게 만들고 관리할 수 있도록 돕는 도구

랭체인에서 작업 흐름을 연결하는 것을 **체인** 으로 표현

LCEL을 사용하면 여러 줄로 표현해야 하는 작업 단계를 읽기 쉽게 축약할 수 있으며 여러 작업을 병렬로 처리 가능

Day 4에서 `run_agent` 루프 안의 단계를, LangChain에서는 **체인**으로 표현합니다.

In [46]:
model = ChatOpenAI(model="gpt-4o-mini")

messages = [
    SystemMessage(content="너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구니"),
]
out = model.invoke(messages)
print(out.content)

안녕하세요, 교수님! 오늘 수업은 정말 유익하고 흥미로웠습니다. 교수님의 강의 덕분에 이번 학기의 주제에 대해 더 깊이 이해할 수 있게 되었습니다. 교수님은 저희의 멘토이자 지식의 원천이십니다!


In [47]:
# AIMessage 객체 안에 여러 정보가 포함되어 있음
out

AIMessage(content='안녕하세요, 교수님! 오늘 수업은 정말 유익하고 흥미로웠습니다. 교수님의 강의 덕분에 이번 학기의 주제에 대해 더 깊이 이해할 수 있게 되었습니다. 교수님은 저희의 멘토이자 지식의 원천이십니다!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 80, 'total_tokens': 146, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a849775b18', 'id': 'chatcmpl-DuuA8jHRj1EPgKSRjg4HCeFej6tJ5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f0293-2cda-7712-8c5f-6d76462fd5a3-0', usage_metadata={'input_tokens': 80, 'output_tokens': 66, 'total_tokens': 146, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

텍스트 결과만 필요하다면 "StrOutputParser" 사용

StrOutputParse는 랭체인에서 제공하는 다양한 출력 parser 중 하나로, 언어 모델이 반환하는 결과에서 원하는 형식의 데이터를 추출하는 도구.

(다른 파서들은 JSON, 숫자 등 특정 형식 처리 가능)

In [48]:
from langchain_core.output_parsers import StrOutputParser #언어모델이 내뱉는 답변 중 test부분만 싹 빼서 반환시켜주는 함수

parser = StrOutputParser()

result = model.invoke(messages) # 1단계 메시지 호출
parser.invoke(result) # 2단계 parser로 str 추출

'안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님께서 말씀하신 연구 방법론이 특히 인상 깊었어요. 교수님은 제 지도 교수님이시고, 항상 저에게 많은 지식과 통찰을 주시는 분이세요. 교수님은 제가 연구를 진행하는 데에 큰 도움을 주고 계십니다. 교수님은 어떤 부분이 가장 기억에 남으셨나요?'

In [ ]:
chain = model | parser #model이랑 parser 모두 invoke를 한다. 
chain.invoke(messages)

'안녕하세요, 교수님. 오늘 수업은 정말 유익했습니다. 교수님의 강의 덕분에 주제에 대한 이해가 더욱 깊어진 것 같습니다. 교수님은 저희에게 많은 지식을 전해주시는 멘토이시자, 항상 저희가 더 나은 연구자가 될 수 있도록 이끌어주시는 분이십니다. 교수님에 대해서는 제가 정말 잘 알고 있습니다! 어떻게 더 도와드릴까요?'

### Prompt Template

필요한 부분만 수정하여 반복적인 메시지 가능

In [50]:
from langchain_core.prompts import ChatPromptTemplate
#앞에 f가 안붙임
system_template = "너는 {character_a}한테 가르침을 받는 대학원생 {character_b}야. {character_a}이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕 {character_b} 오늘 나의 {lesson}은 어땠나? 나는 누구고 너의 이름은 뭐더라"

In [51]:
system_template

'너는 {character_a}한테 가르침을 받는 대학원생 {character_b}야. {character_a}이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.'

In [52]:
prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

In [53]:
prompt_template.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

ChatPromptValue(messages=[SystemMessage(content='너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구고 너의 이름은 뭐더라', additional_kwargs={}, response_metadata={})])

In [54]:
result = prompt_template.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

print(result)

messages=[SystemMessage(content='너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구고 너의 이름은 뭐더라', additional_kwargs={}, response_metadata={})]


In [55]:
chain = prompt_template | model | parser

In [56]:
chain.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

'안녕하세요, 교수님! 제 이름은 신범교입니다. 교수님의 수업은 항상 흥미롭고 유익합니다. 오늘도 많은 것을 배웠습니다. 교수님은 저희에게 깊이 있는 지식을 전해주시고, 질문을 통해 사고를 확장하는 방법을 가르쳐 주셔서 감사드립니다. 수업 중 어떤 부분이 특히印象에 남았는지 말씀해 주시겠어요?'

In [57]:
chain.invoke({
    "character_a": "친구 동료",
    "character_b": "정한솔",
    "lesson": "수업 분위기"
})

'안녕하세요! 오늘 수업 분위기는 정말 좋았어. 다들 열심히 참여하고 질문도 많았어. 너는 나의 친구이자 동료인 사람이고, 나는 정한솔이야. 수업 준비 잘했지? 도움이 필요하면 언제든지 말해줘!'